# triton inference server quickstart

spinning up nvidia triton locally with a tiny model repository. this notebook is mostly setup notes; actual server run is via docker.

## model repo layout

```
models/
  rn18/
    config.pbtxt
    1/
      model.onnx
```

In [ ]:
config_pbtxt = '''
name: "rn18"
platform: "onnxruntime_onnx"
max_batch_size: 8
input [{
  name: "x"
  data_type: TYPE_FP32
  dims: [3, 224, 224]
}]
output [{
  name: "logits"
  data_type: TYPE_FP32
  dims: [1000]
}]
'''
print(config_pbtxt)


## docker run

```
docker run --gpus=all --rm -p8000:8000 -p8001:8001 -p8002:8002 \
  -v $(pwd)/models:/models \
  nvcr.io/nvidia/tritonserver:22.07-py3 tritonserver --model-repository=/models
```

In [ ]:
# pip install tritonclient[http]
# import tritonclient.http as httpclient
# client = httpclient.InferenceServerClient(url='localhost:8000')
# inp = httpclient.InferInput('x', [1,3,224,224], 'FP32')
# inp.set_data_from_numpy(x_np)
# resp = client.infer('rn18', [inp])
# print(resp.as_numpy('logits').shape)


things that bit me: getting the input dims wrong in config.pbtxt, forgetting `max_batch_size` interacts with the leading dim, running the wrong cuda image.

noticed the loss curve flattens after epoch 5. early stop wins.

trying lr=3e-4 vs 1e-3, the higher one diverges on this dataset.

In [ ]:
# add weight decay
# opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
